In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *

In [72]:
import pandas as pd

df = pd.read_csv("titanic.csv")
df


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [74]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

In [73]:
y = df['Survived']
X = df.drop(['Survived'], axis=1)


In [75]:
categorical_features = X.select_dtypes(include=["str"]).columns.tolist()

n_samples = len(X)
n_features = len(X.T)
n_categorical = len(categorical_features)
c_lab, c_num = np.unique(y, return_counts=True)
ir = np.max(c_num) / np.min(c_num)

print("n_samples", n_samples)
print("n_features", n_features)
print("n_categorical", n_categorical)
print("classes", c_lab)
print("class samples", c_num)
print("ir", ir)

n_samples 891
n_features 11
n_categorical 5
classes [0 1]
class samples [549 342]
ir 1.605263157894737


In [76]:
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import accuracy_score

clf = NearestCentroid()

clf.fit(X, y)
y_pred = clf.predict(X)

accuracy = accuracy_score(y, y_pred)

print("Accuracy:", accuracy)


ValueError: Input contains NaN

In [79]:
df.isna()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,False,False,False,False,False,False,False,False,False,False,True,False
1,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,True,False
3,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
886,False,False,False,False,False,False,False,False,False,False,True,False
887,False,False,False,False,False,False,False,False,False,False,False,False
888,False,False,False,False,False,True,False,False,False,False,True,False
889,False,False,False,False,False,False,False,False,False,False,False,False


In [77]:
df.columns[df.isna().any()].tolist()

['Age', 'Cabin', 'Embarked']

In [80]:
print(df["Age"].isna().sum())
print(len(df["Age"]))
print('--')
print(df["Cabin"].isna().sum())
print(len(df["Cabin"]))
print('--')
print(df["Embarked"].isna().sum())
print(len(df["Embarked"]))

177
891
--
687
891
--
2
891


In [81]:
# drop samples
print(len(df))
df = df.dropna(subset=['Embarked'])
print(len(df))

891
889


In [82]:
# drop columns
print(len(df.columns))
df = df.drop(['Cabin'], axis=1)
print(len(df.columns))

12
11


In [83]:
# imputate data
print(df['Age'].mean())
df['Age'] = df['Age'].fillna(df['Age'].median())
print(df['Age'].mean())
print(df["Age"].isna().sum())

29.64209269662921
29.315151856017994
0


In [84]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

y = df['Survived']
X = df.drop(['Survived'], axis=1)

categorical_features = X.select_dtypes(include=["str"]).columns.tolist()

encoder = ColumnTransformer(
    transformers=[("ohe", OneHotEncoder(sparse_output=False), categorical_features)],
    remainder="passthrough",
)

X = encoder.fit_transform(X)

In [85]:
X

array([[ 0.    ,  0.    ,  0.    , ...,  1.    ,  0.    ,  7.25  ],
       [ 0.    ,  0.    ,  0.    , ...,  1.    ,  0.    , 71.2833],
       [ 0.    ,  0.    ,  0.    , ...,  0.    ,  0.    ,  7.925 ],
       ...,
       [ 0.    ,  0.    ,  0.    , ...,  1.    ,  2.    , 23.45  ],
       [ 0.    ,  0.    ,  0.    , ...,  0.    ,  0.    , 30.    ],
       [ 0.    ,  0.    ,  0.    , ...,  0.    ,  0.    ,  7.75  ]],
      shape=(889, 1580))

In [86]:
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import accuracy_score

clf = NearestCentroid()

clf.fit(X, y)
y_pred = clf.predict(X)

accuracy = accuracy_score(y, y_pred)

print("Accuracy:", accuracy)


Accuracy: 0.5995500562429696


## To nie jest prawidłowy sposób weryfikacji klasyfikatora!